# Movie Recommendation System

This notebook uses user-based collaborative filtering to recommend movies for User 2. It finds users with similar rating patterns, filters for movies they rated highly, removes movies User 2 has already rated, and ranks the remaining titles by their average rating.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display


## 1. Load the Data

Place the MovieLens `ratings_small.csv`, `links_small.csv`, and `movies_metadata.csv` files in `MRS/archive/` under the repository root.

In [ ]:
data_dir = Path("MRS/archive")
ratings = pd.read_csv(data_dir / "ratings_small.csv")
movies = pd.read_csv(data_dir / "movies_metadata.csv", low_memory=False)
links = pd.read_csv(data_dir / "links_small.csv")

links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce").astype("Int64")
movies["id"] = pd.to_numeric(movies["id"], errors="coerce").astype("Int64")
movies = movies.dropna(subset=["id"]).drop_duplicates(subset=["id"])


## 2. Connect Ratings to Movie Titles

Ratings use MovieLens `movieId`; the links table maps that key to TMDB IDs, which match the movie metadata `id`.

In [ ]:
movie_ratings = ratings.merge(links, on="movieId", how="left")
movie_ratings = movie_ratings.merge(
    movies[["id", "title"]],
    left_on="tmdbId",
    right_on="id",
    how="left"
)
movie_ratings[["userId", "movieId", "rating", "title"]].head()


## 3. Build the User-Movie Matrix

Rows represent users, columns represent movies, and each cell contains the user?s rating.

In [ ]:
user_movie = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie.head()


## 4. Find Users Similar to User 2

Cosine similarity compares User 2?s rating vector with every other user. The target user is excluded before selecting the five closest matches.

In [ ]:
target_user_id = 2
n_similar_users = 5

rating_matrix = user_movie.fillna(0).to_numpy(dtype=float)
target_position = user_movie.index.get_loc(target_user_id)
target_ratings = rating_matrix[target_position]
denominators = np.linalg.norm(rating_matrix, axis=1) * np.linalg.norm(target_ratings)
similarity_scores = np.divide(
    rating_matrix @ target_ratings,
    denominators,
    out=np.zeros_like(denominators),
    where=denominators != 0
)

similar_positions = np.argsort(similarity_scores)[::-1]
similar_positions = similar_positions[similar_positions != target_position][:n_similar_users]
similar_user_ids = user_movie.index[similar_positions]
similar_users = pd.DataFrame({
    "userId": similar_user_ids,
    "similarity": similarity_scores[similar_positions]
})

display(similar_users)


## 5. Rank Unwatched Recommendations

Keep neighbor ratings of at least 4.0, remove every movie User 2 has already rated, then calculate the mean rating for each remaining movie.

In [ ]:
min_rating = 4.0
n_recommendations = 20

neighbor_ratings = (
    user_movie.loc[similar_user_ids]
    .stack()
    .rename("rating")
    .reset_index()
)
highly_rated = neighbor_ratings[neighbor_ratings["rating"] >= min_rating]
watched_movie_ids = user_movie.columns[user_movie.loc[target_user_id].notna()]
unwatched_ratings = highly_rated[
    ~highly_rated["movieId"].isin(watched_movie_ids)
]

movie_scores = (
    unwatched_ratings.groupby("movieId", as_index=False)
    .agg(
        mean_rating=("rating", "mean"),
        similar_user_count=("rating", "count")
    )
)
movie_catalog = (
    movie_ratings[["movieId", "title"]]
    .dropna(subset=["title"])
    .drop_duplicates("movieId")
)
recommendations = (
    movie_scores.merge(movie_catalog, on="movieId", how="left")
    .sort_values(
        ["mean_rating", "similar_user_count"],
        ascending=[False, False]
    )
    .head(n_recommendations)
    .reset_index(drop=True)
)


## 6. Top Movie Recommendations

The results are ordered by average neighbor rating, with the number of similar users who recommended each movie used as a tie-breaker.

In [ ]:
display(recommendations[["title", "mean_rating", "similar_user_count"]])
